In [ ]:
import os
from PIL import Image, ImageDraw
from ultralytics import YOLO
import numpy as np

# 載入 yolov8 模型
yolo = YOLO('./yolov8m.pt')

# 資料夾路徑
input_folder_path = './dataset/'
output_folder_path = './pre/'

# 獲取資料夾中所有檔案
files = os.listdir(input_folder_path)

# 迴圈處理每個檔案
for filename in files:
    # 檔案完整路徑
    input_file_path = os.path.join(input_folder_path, filename)

    # 提取年齡、性別等相關信息
    parts = filename.split('_')
    age = parts[0]
    gender = parts[1]

    # 構建標註信息
    annotation = {'age': age, 'gender': gender}

    # 讀取圖片
    print(f"1-1.Opening image: {input_file_path}")
    img = Image.open(input_file_path)
    print(f"1-2.Image opened successfully.")

    # 使用 YOLO 模型進行物件檢測
    print("2-1.Predicting with YOLO model.")
    results = yolo.predict(img)
    print("2-2.Prediction completed.")

    # 保存 bbox 資訊的列表
    bbox_list = []

    # 在圖片上繪製人臉框
    draw = ImageDraw.Draw(img)

    for detection in results[0].boxes.data.numpy():
        bbox = detection[0:4].tolist()
        class_id = int(detection[5])
        x, y, w, h = bbox
        # 將相對座標轉換為相對尺寸
        x, w = x + w / 2, w
        y, h = y + h / 2, h
        # 寫入標籤信息
        label_line = f"{class_id} {x} {y} {w} {h} {age} {gender}\n"
        bbox_list.append(label_line)

        # 繪製 bbox
        draw.rectangle(bbox, outline="red", width=2)

    # 保存標註後的圖片
    output_annotated_image_path = os.path.join(output_folder_path, f"{os.path.basename(filename)}")
    print(f"3.Save img: {output_annotated_image_path}")
    img.save(output_annotated_image_path)

    # 保存標籤文件
    output_label_file_path = os.path.join(output_folder_path, f"{os.path.splitext(filename)[0]}.txt")
    with open(output_label_file_path, 'w') as f:
        for detection in results[0].boxes.data.numpy():
            bbox = detection[0:4].tolist()
            class_id = int(detection[5])
            x, y, w, h = bbox
            # 將相對座標轉換為相對尺寸
            x, w = x + w / 2, w
            y, h = y + h / 2, h
            # 寫入標籤信息
            id = str(class_id) + "_" + gender + "_" + age
            label_line = f"{id} {x} {y} {w} {h}\n"
            f.write(label_line)

    print("Label file saved./n-------------------------------")
